In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# PLBART + PRIMEVUL : ONE-CELL FINAL OPTIMIZED CODE (KAGGLE)
# ============================================================

# --------------------
# Imports
# --------------------
import os
import json
import torch
import numpy as np
import pandas as pd
from datetime import datetime

from datasets import Dataset
from transformers import (
    PLBartTokenizer,
    PLBartForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------
# DEVICE
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# DATASET PATH
# ============================================================

DATASET_PATH = "/kaggle/input/dataset-primevulplb"

print("\nFiles in dataset:")
for f in os.listdir(DATASET_PATH):
    print(" -", f)

# ============================================================
# LOAD PRIMEVUL JSONL
# ============================================================

def load_primevul_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line.strip())
                data.append({
                    "code": obj["func"],
                    "label": int(obj["target"])
                })
            except Exception:
                continue
    return pd.DataFrame(data)

print("\nLoading PrimeVul dataset...")

train_df = load_primevul_jsonl(f"{DATASET_PATH}/primevul_train_paired.jsonl")
val_df   = load_primevul_jsonl(f"{DATASET_PATH}/primevul_valid_paired.jsonl")
test_df  = load_primevul_jsonl(f"{DATASET_PATH}/primevul_test_paired.jsonl")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("\nTrain label distribution:\n", train_df["label"].value_counts())

# ============================================================
# HF DATASETS
# ============================================================

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df, preserve_index=False)
test_ds  = Dataset.from_pandas(test_df, preserve_index=False)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = PLBartTokenizer.from_pretrained("uclanlp/plbart-base")

def tokenize_fn(batch):
    return tokenizer(
        batch["code"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch", columns=cols)
test_ds.set_format("torch", columns=cols)

# ============================================================
# MODEL
# ============================================================

model = PLBartForSequenceClassification.from_pretrained(
    "uclanlp/plbart-base",
    num_labels=2
).to(device)

# ============================================================
# TRAINING ARGUMENTS (OPTIMIZED)
# ============================================================

training_args = TrainingArguments(
    output_dir="./plbart_primevul",
    per_device_train_batch_size=8,      # stable for PLBART
    per_device_eval_batch_size=8,
    num_train_epochs=6,                 # best tradeoff
    learning_rate=1e-5,                 # 🔥 lower LR = better convergence
    weight_decay=0.01,
    fp16=True,
    warmup_ratio=0.1,                   # smoother loss curve
    save_strategy="no",                 # 🔥 avoid disk crash
    logging_steps=100,
    report_to="none"
)

# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

# ============================================================
# TRAIN
# ============================================================

print("\nStarting PLBART training on PrimeVul...")
trainer.train()

# ============================================================
# FINAL EVALUATION
# ============================================================

print("\nEvaluating on test set...")

preds = trainer.predict(test_ds)

logits = preds.predictions
if isinstance(logits, tuple):   # PLBART safety
    logits = logits[0]

y_true = preds.label_ids
y_pred = np.argmax(logits, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

print("\n===== FINAL PLBART PRIMEVUL RESULTS =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("FPR      :", fpr)
print("Confusion Matrix:", tn, fp, fn, tp)
print("Prediction distribution:", np.unique(y_pred, return_counts=True))

# ============================================================
# SAVE MINIMAL RESULTS (LOW DISK SAFE)
# ============================================================

results = {
    "dataset": "PrimeVul",
    "model": "PLBART",
    "epochs": 6,
    "learning_rate": 1e-5,
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "fpr": float(fpr),
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    },
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

out_path = "/kaggle/working/PLBART_PrimeVul_metrics.json"
with open(out_path, "w") as f:
    json.dump(results, f)

print("\nResults saved to:", out_path)


2026-01-30 20:13:35.614603: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769804016.065438      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769804016.192836      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769804017.296207      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769804017.296252      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769804017.296255      55 computation_placer.cc:177] computation placer alr

Device: cuda
GPU: Tesla T4

Files in dataset:
 - primevul_test_paired.jsonl
 - primevul_train_paired.jsonl
 - primevul_valid_paired.jsonl

Loading PrimeVul dataset...
Train: (7578, 2)
Val  : (960, 2)
Test : (870, 2)

Train label distribution:
 label
1    3789
0    3789
Name: count, dtype: int64


sentencepiece.bpe.model:   0%|          | 0.00/986k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/783 [00:00<?, ?B/s]

Map:   0%|          | 0/7578 [00:00<?, ? examples/s]

...


Starting PLBART training on PrimeVul...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,0.715700
200,0.716300
300,0.716600
400,0.710600
500,0.700700
600,0.698800
700,0.699400
800,0.699600
900,0.698900
1000,0.697700



Evaluating on test set...



===== FINAL PLBART PRIMEVUL RESULTS =====
Accuracy : 0.5149425287356322
Precision: 0.5226480836236934
Recall   : 0.3448275862068966
F1 Score : 0.4155124653739612
FPR      : 0.31494252873563217
Confusion Matrix: 298 137 285 150
Prediction distribution: (array([0, 1]), array([583, 287]))

Results saved to: /kaggle/working/PLBART_PrimeVul_metrics.json
